# Downloading Planet imagery using the Planet API

This notebook is for downloading Planet images from a single day, i.e. without compositing (creating a mosaic). For downloading Planet multiple images from a date range that are turned into a mosaic (compositing), I suggest using the Planet online platform directly, as the quality of the images can be manually checked, and it gives more control over the final image.

There are two main imagery filters to select by, which are the 'area_coverage' which is how much of the area of interest (AOI) is covered by the image, and the 'cloud_coverage' which is how much of the image is covered by clouds. When selecting by these filters, full images will either be included if they fit the criteria, or excluded if they don't. As far as I can tell, it isn't possible to filter at a pixel-level on the Planet side. But, it is possible to download all images for the date range and AOI, and then use the udm2 quality assurance layer to filter out the pixels that are not usable. We do not explore that option in this notebook.

## Import functions from planet_utils.py

In [24]:
import os
import json
import pandas as pd

import folium

from datetime import datetime, timedelta

# Planet API imports
from planet import Auth, Session, DataClient, OrdersClient, data_filter
from planet.order_request import build_request, product

import counting_boats.boat_utils.planet_utils as planet_utils
# import counting_boats.boat_utils.auto_helpers as auto_helpers

### Select the polygon of interest

This should be a geojson file

In [25]:
polygon_name = 'example'
polygon_directory = 'data/polygons'
selected_polygon = f'{polygon_directory}/{polygon_name}.geojson'

print(f'Polygon: {selected_polygon}')

# does the polygon exist?
if not os.path.exists(selected_polygon):
    print(f'Polygon {selected_polygon} does not exist')
    exit(1)

Polygon: data/polygons/example.geojson


In [26]:
# Load GeoJSON from file
with open(selected_polygon, 'r') as f:
    geojson_data = json.load(f)

# Extract the geometry from the GeoJSON
# Depending on your GeoJSON structure, you might need one of these approaches:
if 'geometry' in geojson_data:
    # For a Feature
    aoi = geojson_data['geometry']
elif 'type' in geojson_data and geojson_data['type'] == 'FeatureCollection':
    # For a FeatureCollection, use the first feature's geometry
    aoi = geojson_data['features'][0]['geometry']
elif 'coordinates' in geojson_data and 'type' in geojson_data:
    # If it's already a bare geometry
    aoi = geojson_data

# Now use the geometry object with the filter
aoi_filter = data_filter.geometry_filter(aoi)

print(aoi)

{'coordinates': [[[134.50075520473314, -12.972551552526141], [134.50075520473314, -13.138249590118846], [134.75653574094503, -13.138249590118846], [134.75653574094503, -12.972551552526141], [134.50075520473314, -12.972551552526141]]], 'type': 'Polygon'}


In [27]:
# Calculate the center of the polygon for map centering
lats = [coord[1] for coord in aoi['coordinates'][0]]
lons = [coord[0] for coord in aoi['coordinates'][0]]
center_lat = sum(lats) / len(lats)
center_lon = sum(lons) / len(lons)

# Create the map centered on your polygon
m = folium.Map(location=[center_lat, center_lon], zoom_start=8)
folium.TileLayer('Esri.WorldImagery').add_to(m)  # Satellite imagery

# Create a proper GeoJSON Feature for folium
geojson_feature = {
    'type': 'Feature',
    'properties': {},
    'geometry': aoi
}

# Add your polygon to the map with some styling
folium.GeoJson(
    geojson_feature,
    name='polygon',
    style_function=lambda x: {
        'fillColor': '#3388ff',
        'color': '#3388ff',
        'weight': 2,
        'fillOpacity': 0.4,
    }
).add_to(m)

# Add layer control to toggle basemaps
folium.LayerControl().add_to(m)

display(m)

### Set output directory for planet downloads (tif files)

And create if it doesn't exist.

In [28]:
# output directory for saving images
output_dir = f'images/RawImages'
os.makedirs(output_dir, exist_ok=True)

### Select the date range of interest

We have a date that we want, and we subtract and add a few days to get a range that we can query planet for.

In [29]:
# Given date as a string
selected_date = '2024-12-15'

# Number of days either side of the date to filter by
days_range = 14

# Convert string to datetime object
selected_date_obj = datetime.strptime(selected_date, '%Y-%m-%d')

# Subtract some time to get images from within the month
lower_date = selected_date_obj - timedelta(days=days_range)
# Add days
upper_date = selected_date_obj + timedelta(days=days_range)

# Convert back to string if needed
lower_date_str = lower_date.strftime('%Y-%m-%d')
upper_date_str = upper_date.strftime('%Y-%m-%d')

print(f'Lower date: {lower_date_str}')
print(f'Upper date: {upper_date_str}')

Lower date: 2024-12-01
Upper date: 2024-12-29


## Using the functions directly from plant_utils.py

Change the cloud cover argument here. Only images with LESS than this cloud cover will be returned.

In [32]:
polygon_search = planet_utils.PlanetSearch(
    polygon_file=selected_polygon,
    min_date=lower_date_str,
    max_date=upper_date_str,
    cloud_cover=0.25,
)

print(f"Number of images found in search: {len(polygon_search)}")

# # extract image IDs only
# image_ids = [feature['id'] for feature in polygon_search]
# print(image_ids)

Number of images found in search: 24


## Select the images from the above images

Here is where the area_coverage argument is set. This is the percentage of the image that is covered by the AOI. Any GREATER than this value will be returned.

In [35]:
dates = pd.date_range(start=lower_date_str, end=upper_date_str).strftime("%Y-%m-%d")
# print(dates)

# Select images for each date and return them
items = []

for date in dates:
    try:
        it = planet_utils.PlanetSelect(
            items=polygon_search,
            polygon=selected_polygon,
            date=date,
            area_coverage=0.95,
        )
    except Exception as e:
        # traceback.print_exc()
        print(e)
        continue
    if it is None or len(it) == 0:
        continue
    items.append(it)

print(f"Total images: {len(items)}")  # This shows how many dates have valid images

Total images: 3


### Extract the image IDs, which shows the date

In [41]:
# Extract IDs correctly from the nested structure
image_ids = []
for date_items in items:
    for feature in date_items:
        image_ids.append(feature['id'])

print(image_ids)

['20241204_013306_68_24fa', '20241204_013304_60_24fa', '20241204_013302_53_24fa', '20241204_013225_29_251a', '20241204_013221_08_251a', '20241204_013223_18_251a', '20241216_013143_50_24d7', '20241216_013141_26_24d7', '20241221_013322_40_24f2', '20241221_013320_16_24f2', '20241221_013324_63_24f2', '20241221_004852_93_24a8', '20241221_004848_91_24a8', '20241221_004850_92_24a8']


### Extract the dates

In [40]:
for i in items:
    print(i[0]["properties"]["acquired"][:10])

2024-12-04
2024-12-16
2024-12-21


# Place an order for the images from Planet

You will need to ensure that you have the planet API key set in your environment variables. This needs to be set in the config.yml file, which the PlanetOrder function calls. It can either be set in the config.yml file, or in the environment variables. 

In the config.yml there is this code:

```python
planet:  
  api_key:
    'ENV'  
    # If api_key is 'ENV', there must be a '.env' file in the same directory as this file with PLANET_API_KEY={your_api_key}
```

Which means that you should have a .env file in the same directory as this file with the following line (which has the actual api key within the curly brackets):

```text
PLANET_API_KEY={your_api_key} 
```

In my directory the file is called api_key.env.

We also need to select one of the images from the above dates (if there are more than one). We can check the area coverage and cloud coverage of the images that we matched with, and choose one based on that.

In [ ]:
for i in range(len(items)):

    print(f"Image {i}")

    # Date of the image
    image_date = items[i][0]["properties"]["acquired"][:10]
    print(f"\nDate of Planet image: {image_date}")

    # Calculate the area coverage of the items in the polygon
    area_cov = planet_utils.items_area_coverage(
        items=items[i],
        AOI=selected_polygon
    )

    print(f"Area coverage: {area_cov}")

    # How many sub-images are there?
    n_sub_images = len(items[i])

    # What is their cloud cover?
    for j in range(n_sub_images):
        print(f"Sub-image {j} cloud cover: {items[i][j]['properties']['cloud_cover']}")

    print("\n\n")

Image 0

Date of Planet image: 2024-12-04
Area coverage: 0.9999999999999983
Sub-image 0 cloud cover: 0
Sub-image 1 cloud cover: 0.01
Sub-image 2 cloud cover: 0.01
Sub-image 3 cloud cover: 0.02
Sub-image 4 cloud cover: 0.04
Sub-image 5 cloud cover: 0.04



Image 1

Date of Planet image: 2024-12-16
Area coverage: 0.9999999999999983
Sub-image 0 cloud cover: 0.05
Sub-image 1 cloud cover: 0.18



Image 2

Date of Planet image: 2024-12-21
Area coverage: 0.9999999999999983
Sub-image 0 cloud cover: 0
Sub-image 1 cloud cover: 0
Sub-image 2 cloud cover: 0
Sub-image 3 cloud cover: 0
Sub-image 4 cloud cover: 0
Sub-image 5 cloud cover: 0





### Place the order

This function should only be run once.

In [ ]:
# Choose one of the images as an index
image_choice = 2

# Date of the ordered item
date = items[image_choice][0]["properties"]["acquired"][:10]
print(f"Date of chosen image: {date}")

fs_date = "".join(date.split("-"))  # filesafe date

try:
    order = planet_utils.PlanetOrder(
        polygon_file=selected_polygon, 
        items=items[0], 
        name=f"{polygon_name}_{fs_date}"
    )
except Exception as e:
    # traceback.print_exc()
    print(e)
    # return ""

# print(order)

print(order['id'])
order_id = order['id']

Date of chosen image: 2024-12-21
{'_links': {'_self': 'https://api.planet.com/compute/ops/orders/v2/95e62a7b-e497-4886-88f9-afd4b7e21a65'}, 'created_on': '2025-03-31T01:19:52.106688Z', 'delivery': {'archive_filename': 'planet_image_api', 'archive_type': 'zip', 'single_archive': True}, 'error_hints': [], 'id': '95e62a7b-e497-4886-88f9-afd4b7e21a65', 'last_message': 'Preparing order', 'last_modified': '2025-03-31T01:19:52.106688Z', 'metadata': {'stac': {}}, 'name': 'example_20241221', 'notifications': {}, 'order_type': 'partial', 'products': [{'item_ids': ['20241204_013306_68_24fa', '20241204_013304_60_24fa', '20241204_013302_53_24fa', '20241204_013225_29_251a', '20241204_013221_08_251a', '20241204_013223_18_251a'], 'item_type': 'PSScene', 'product_bundle': 'analytic_sr_udm2'}], 'state': 'queued', 'tools': [{'clip': {'aoi': {'coordinates': [[[134.50075520473314, -12.972551552526141], [134.50075520473314, -13.138249590118846], [134.75653574094503, -13.138249590118846], [134.75653574094503

## Check the planet order

It must read 'success' for the images to be downloaded.

In [120]:
planet_utils.PlanetCheckOrder(order_id)

'running'

Once the orders have been completed, they can be downloaded to the output directory. 

However, it takes a while for the orders to be completed (at least 15 mins, can be more than an hour for large images), so other AOIs and dates can be ordered by going over the steps above again.

If you created multiple orders, then you can create a list of the orders and download them all at once.

### Create a list of orders

You will need to create some criteria to match the orders. I've used the date that the order was placed. I have it checking the orders that were placed on today's date, but that can be changed by manually setting a date, and then use the code below to download the list of orders.

In [119]:
# get today's date for ordering
order_date = datetime.now().strftime("%Y-%m-%d")
# order_date = "2025-03-31" # to use a different order date for downloading

# check the order status of all previous orders
all_orders = planet_utils.get_orders()
# print(all_orders)

# Count running orders
running_orders = [o for o in all_orders if o["state"] == "running"]
print(f"{len(running_orders)} orders are still running")

# Count completed orders
completed_orders = [o for o in all_orders if o["state"] == "success" and o['last_modified'][:10] == order_date]
print(f"{len(completed_orders)} orders from {order_date} are ready")

1 orders are still running
1 orders from 2025-03-31 are ready


## Download the images (when ready)

It is possible to download a single order by using the order_id, or all orders by using the completed_orders list. 

The `PlanetDownload` function also unzips the zip files, and places the tif files in the output directory, which should be in the correct naming format.

### To download a single order

In [ ]:
planet_utils.PlanetDownload(
        orderID=order_id,
        aoi=polygon_name,
        date=order['products'][0]['item_ids'][0][:8], 
        downloadPath=output_dir)

### To download all orders in the `completed_orders` list

Modify the list arguments as necessary above (such as the `order_date`).

In [82]:
# Count completed orders
completed_orders = [o for o in all_orders if o["state"] == "success" and o['last_modified'][:10] == order_date]

print(f"Downloading {len(completed_orders)} orders from {order_date}")

for order in completed_orders:
    # download the images
    planet_utils.PlanetDownload(
        orderID=order['id'],
        # aoi=order['name'][:10],
        aoi=polygon_name,
        date=order['products'][0]['item_ids'][0][:8], 
        downloadPath=output_dir)

images/RawImages\example_20241204.zip images/RawImages\example_20241204


For downloading composite images, I suggest using the Planet platform, as the quality of the images can be manually checked, and it gives more control over the final image.

End of the script. 